# TP5 — Introduction à ClickHouse

**ClickHouse** est une base de données **orientée colonnes** conçue pour l'analytique (**OLAP**).

| | MySQL (row-oriented) | ClickHouse (column-oriented) |
|--|--|--|
| Cas d'usage | Transactions (OLTP) | Analytique (OLAP) |
| Agrégations | Lent sur gros volumes | Très rapide |
| UPDATE/DELETE | Natif | Limité (mutations) |

**Tables disponibles** dans `tp_clickhouse` :
- `web_logs` — Logs serveur web
- `iot_metrics` — Capteurs IoT
- `ecommerce_events` — Événements e-commerce

## 1. Initialisation

In [ ]:
!pip install clickhouse-connect pandas seaborn matplotlib -q

In [ ]:
import clickhouse_connect
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

client = clickhouse_connect.get_client(host='clickhouse', port=8123,
                                       username='default', password='clickhouse')

def query(sql):
    """Exécute une requête SQL et retourne un DataFrame."""
    r = client.query(sql)
    return pd.DataFrame(r.result_rows, columns=r.column_names)

print('ClickHouse', client.command('SELECT version()'))

---
## 2. Exploration de la base

In [ ]:
query('SHOW TABLES FROM tp_clickhouse')

Observer les types ClickHouse : `LowCardinality`, `Enum8`, `UInt32`, `DateTime`…

In [ ]:
query('DESCRIBE tp_clickhouse.web_logs')

In [ ]:
query('SELECT * FROM tp_clickhouse.web_logs LIMIT 5')

In [ ]:
for t in ['web_logs', 'iot_metrics', 'ecommerce_events']:
    print(f"{t}: {client.command(f'SELECT count() FROM tp_clickhouse.{t}')} lignes")

### Exercice 1
Affichez la structure (`DESCRIBE`) et les 5 premières lignes de `ecommerce_events`.

In [ ]:
# À compléter

---
## 3. Le moteur MergeTree

MergeTree est le moteur principal de ClickHouse :
- **`ORDER BY`** — tri physique sur disque, accélère les filtres
- **`PARTITION BY`** — découpe par date, élagage automatique
- **`TTL`** — suppression automatique des données expirées

In [ ]:
print(client.command('SHOW CREATE TABLE tp_clickhouse.web_logs'))

In [ ]:
query("""
SELECT table, partition, rows, formatReadableSize(bytes_on_disk) AS size
FROM system.parts
WHERE database = 'tp_clickhouse' AND active
ORDER BY table, partition
""")

---
## 4. Requêtes analytiques — Web logs

ClickHouse utilise un SQL standard enrichi de fonctions analytiques.

### Temps de réponse moyen par pays

In [ ]:
query("""
SELECT country, count() AS nb,
       round(avg(response_time_ms), 1) AS avg_ms,
       max(response_time_ms) AS max_ms
FROM tp_clickhouse.web_logs
GROUP BY country
ORDER BY nb DESC
""")

### Requêtes par heure — fonctions temporelles

Fonctions utiles : `toHour()`, `toStartOfDay()`, `toStartOfMonth()`, `toYYYYMM()`

In [ ]:
df_h = query("""
SELECT toHour(timestamp) AS heure, count() AS nb
FROM tp_clickhouse.web_logs
GROUP BY heure ORDER BY heure
""")

sns.barplot(data=df_h, x='heure', y='nb', color='steelblue')
plt.title('Requêtes par heure')
plt.tight_layout()
plt.show()

### `countIf` / `sumIf` — agrégations conditionnelles

In [ ]:
query("""
SELECT toYYYYMM(timestamp) AS mois,
       count() AS total,
       countIf(status_code >= 400) AS erreurs,
       round(countIf(status_code >= 400) * 100.0 / count(), 1) AS taux_erreur_pct
FROM tp_clickhouse.web_logs
GROUP BY mois ORDER BY mois
""")

### Exercice 2
Calculez le nombre de requêtes et le volume total (`sum(bytes_sent)`) par **navigateur** (`browser`). Triez par volume décroissant.

In [ ]:
# À compléter

### Exercice 3
Comparez les performances **mobile vs desktop** (`is_mobile`) : nombre de requêtes, temps de réponse moyen, taux d'erreur.

In [ ]:
# À compléter

---
## 5. Séries temporelles — IoT

ClickHouse excelle sur les séries temporelles grâce à ses fonctions de date et ses agrégations optimisées.

In [ ]:
df_paris = query("""
SELECT timestamp, temperature, humidity
FROM tp_clickhouse.iot_metrics
WHERE device_id = 'sensor_001'
ORDER BY timestamp
""")

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(df_paris['timestamp'], df_paris['temperature'], 'b-o', markersize=3, label='Temp °C')
ax1.set_ylabel('Température (°C)', color='blue')
ax2 = ax1.twinx()
ax2.plot(df_paris['timestamp'], df_paris['humidity'], 'r-s', markersize=3, alpha=0.6, label='Humidité %')
ax2.set_ylabel('Humidité (%)', color='red')
plt.title('Capteur Paris — Température & Humidité')
plt.tight_layout()
plt.show()

In [ ]:
query("""
SELECT toStartOfDay(timestamp) AS jour, device_id,
       round(avg(temperature), 2) AS avg_temp,
       round(avg(humidity), 2) AS avg_hum
FROM tp_clickhouse.iot_metrics
WHERE device_type = 'temperature'
GROUP BY jour, device_id
ORDER BY device_id, jour
""")

### Exercice 4
Calculez la température moyenne par **mois** et par **ville** (`location`). Utilisez `toStartOfMonth()`.

In [ ]:
# À compléter

---
## 6. Analyse e-commerce — Funnel

Le funnel classique : `view → cart → purchase`. On utilise `countIf()` et `sumIf()` pour l'analyser.

In [ ]:
df_ev = query("""
SELECT event_type, count() AS nb
FROM tp_clickhouse.ecommerce_events
GROUP BY event_type ORDER BY nb DESC
""")

sns.barplot(data=df_ev, x='event_type', y='nb',
            order=['view','cart','purchase','refund'], palette='Blues_r')
plt.title('Funnel e-commerce')
plt.tight_layout()
plt.show()

In [ ]:
query("""
SELECT category,
       countIf(event_type = 'view') AS vues,
       countIf(event_type = 'purchase') AS achats,
       round(sumIf(price, event_type = 'purchase'), 2) AS CA,
       round(countIf(event_type = 'purchase') * 100.0 / countIf(event_type = 'view'), 1) AS conversion_pct
FROM tp_clickhouse.ecommerce_events
GROUP BY category
ORDER BY CA DESC
""")

### Exercice 5
Calculez le chiffre d'affaires par **marque** (`brand`), uniquement pour les achats. Triez par CA décroissant.

In [ ]:
# À compléter

### Exercice 6
Calculez le **CA net** par mois (`toStartOfMonth`) : achats − remboursements. Utilisez `sumIf()`.

In [ ]:
# À compléter

---
## 7. Fonctions avancées

- **`quantile(0.9)(col)`** — percentiles
- **`groupArray(col)`** — agrège en tableau
- **`multiIf(cond1, val1, …, default)`** — conditions multiples

In [ ]:
query("""
SELECT quantile(0.5)(response_time_ms) AS p50,
       quantile(0.9)(response_time_ms) AS p90,
       quantile(0.99)(response_time_ms) AS p99
FROM tp_clickhouse.web_logs
""")

In [ ]:
query("""
SELECT country,
       groupArray(DISTINCT browser) AS navigateurs
FROM tp_clickhouse.web_logs
GROUP BY country
ORDER BY length(groupArray(DISTINCT browser)) DESC
""")

### Exercice 7
Classifiez les requêtes web en catégories de performance avec `multiIf()` :
- < 100 ms → `rapide`
- < 500 ms → `normal`
- < 2000 ms → `lent`
- sinon → `très lent`

Affichez le nombre de requêtes et le temps moyen par catégorie.

In [ ]:
# À compléter

---
## Résumé

| Concept | Ce qu'on a vu |
|---------|---------------|
| Architecture | Stockage colonnes, MergeTree, partitions |
| Types | LowCardinality, Enum, UInt, DateTime |
| Analytique | GROUP BY, countIf, sumIf, quantile |
| Séries temporelles | toHour, toStartOfDay, toStartOfMonth |
| Fonctions avancées | groupArray, multiIf |

[Documentation ClickHouse](https://clickhouse.com/docs) · [Playground](https://play.clickhouse.com/)